# Figure 7 -- position error and matching rates

Forecast eddies are matched one-to-one to observed eddies of the same polarity.
Three quantities are followed with lead day: the median centre-to-centre separation
of the matched pairs, the percentage of observed eddies that are matched, and the
percentage of forecast eddies left unmatched.

Colour denotes polarity here rather than product, because these quantities belong
to the match and not to either product on its own.

**What this reproduces.** The whole figure. Every variable is a distance or a
ratio, so nothing in it had to be withheld.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
POL_COLOR = {"anticyclonic": "#C0392B", "cyclonic": "#1B6CA8"}
POL_MARK  = {"anticyclonic": "o",       "cyclonic": "s"}
ROWS = [("median_separation", "Median separation (km)"),
        ("correct_fraction",  "Correct forecasts (%)"),
        ("false_fraction",    "False eddies (%)")]
SHOW_PAIR_COUNT = True

path = need("Figure7_*.nc")[0]
with xr.open_dataset(path) as ds:
    fig, axes = plt.subplots(len(ROWS), ds.sizes["region"],
                             figsize=(7.5, 1.55 * len(ROWS) + 0.6),
                             dpi=130, sharex=True, squeeze=False)
    for i, (v, ylab) in enumerate(ROWS):
        lo = float(np.nanmin((ds[v] - ds[v + "_se"]).values))
        hi = float(np.nanmax((ds[v] + ds[v + "_se"]).values))
        pad = 0.08 * (hi - lo if hi > lo else max(abs(hi), 1.0))
        for j, rg in enumerate(ds.region.values):
            ax = axes[i][j]
            for po in ds.polarity.values:
                m  = ds[v].sel(region=rg, polarity=po)
                se = ds[v + "_se"].sel(region=rg, polarity=po)
                ax.plot(ds.lead_day, m, ls="-", marker=POL_MARK[str(po)],
                        ms=2.5, lw=1.4, color=POL_COLOR[str(po)],
                        label=str(po).capitalize() if (i == 0 and j == 0) else None)
                ax.fill_between(ds.lead_day, m - se, m + se,
                                color=POL_COLOR[str(po)], alpha=0.18, lw=0)
            ax.set_ylim(lo - pad, hi + pad)
            ax.grid(ls="--", lw=0.3)
            ax.tick_params(labelsize=7)
            ax.set_xticks(np.arange(1, 16, 2))
            ax.set_xlim(0.5, 15.5)
            ax.text(0.03, 0.94, f"({chr(97 + i * ds.sizes['region'] + j)})",
                    transform=ax.transAxes, fontsize=7.5, fontweight="bold",
                    va="top")
            if i == 0:
                ax.set_title(str(rg), fontsize=8.5, fontweight="bold")
                if SHOW_PAIR_COUNT:
                    ax2 = ax.twinx()
                    for po in ds.polarity.values:
                        ax2.plot(ds.lead_day,
                                 ds.n_pairs.sel(region=rg, polarity=po),
                                 ls=":", lw=0.9, color=POL_COLOR[str(po)])
                    ax2.set_ylim(bottom=0)
                    ax2.tick_params(labelsize=6)
                    if j == ds.sizes["region"] - 1:
                        ax2.set_ylabel("matched pairs", fontsize=6)
            if j == 0:
                ax.set_ylabel(ylab, fontsize=7)
            if i == len(ROWS) - 1:
                ax.set_xlabel("Forecast lead day", fontsize=7.5)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), fontsize=7.5,
               frameon=False, bbox_to_anchor=(0.5, -0.015))
    fig.tight_layout(h_pad=0.4, w_pad=0.5, rect=(0, 0.03, 1, 1))
    out = FIGDIR / "Figure7_matching.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", out)

    print("\nranges across regions and polarities")
    for nm in ("median_separation", "correct_fraction", "false_fraction"):
        d1, d15 = ds[nm].sel(lead_day=1).values, ds[nm].sel(lead_day=15).values
        print(f"  {nm:20s} day 1 {np.nanmin(d1):6.1f}-{np.nanmax(d1):6.1f}"
              f"   day 15 {np.nanmin(d15):6.1f}-{np.nanmax(d15):6.1f}")
